# 10 — Local DINOv2 full five-fold inference

Notebook này tiếp nối Phase 2A (`08_local_dinov2_preprocessing.ipynb`) và Phase 2B (`09_local_dinov2_model_restore.ipynb`).

**Mục tiêu Phase 2C:** chạy full-study inference trên toàn bộ valid 2.5D windows của một study, lần lượt restore 5 fold checkpoints, lấy 12 sigmoid probabilities từ mỗi fold, rồi tính trung bình xác suất qua 5 folds.

Nguồn chuẩn là notebook Kaggle cuối `08_model_ensemble_kaggle_dataset_ready.ipynb`:
- Cell 29: `FiveRoleDINOv2_2p5D` và `forward()`.
- Cell 31: instantiate local DINOv2 source, `pretrained=False`, restore checkpoints bằng `strict=True`.
- Cell 33: `sigmoid(logits.float())` từng fold và `fold_probabilities.mean(axis=1)`.

**Khác biệt triển khai local:** chỉ giữ một fold model trong memory tại một thời điểm. Quy tắc dự đoán không đổi; cách này hạn chế RAM trên laptop. CPU mặc định dùng encoder chunk 2; có thể giảm còn 1 nếu thiếu RAM. Không training, không sửa notebook competition, không blend với ViT trong phase này.

> Study mẫu xuất phát từ `gold_validation`, chỉ dùng kiểm thử phần mềm. Prediction ở đây không phải đánh giá độc lập hoặc điểm validation mới.

## 1. Project paths, imports and runtime

In [1]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from rsna_knee.models.dinov2_bundle import (
    find_bundle_root,
    resolve_bundle_root,
    validate_bundle_root,
)
from rsna_knee.models.dinov2_five_role import FIVE_ROLE_ORDER, TARGETS
from rsna_knee.inference.dinov2_ensemble import (
    run_dinov2_five_fold_inference,
    save_dinov2_inference_result,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODER_CHUNK_SIZE = 8 if DEVICE.type == "cuda" else 2
USE_AMP = DEVICE.type == "cuda"  # Mirrors Kaggle's AMP behavior on CUDA; always off on CPU.

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Encoder chunk size:", ENCODER_CHUNK_SIZE)
print("AMP enabled:", USE_AMP)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA unavailable: full five-fold inference will run on CPU.")

Project root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection
Python: 3.13.9
PyTorch: 2.11.0+cpu
Device: cpu
Encoder chunk size: 2
AMP enabled: False
CUDA unavailable: full five-fold inference will run on CPU.


## 2. Load the Phase 2A full-study preprocessing output

This is the full `[5, 10, 3, 224, 224]` study tensor, not the five-window smoke-test mask used in Phase 2B.

In [2]:
PREPROCESS_DIR = PROJECT_ROOT / "demo" / "output" / "dinov2_preprocessing"
PREPROCESSED_PATH = PREPROCESS_DIR / "dinov2_preprocessed_sample.npz"
SUMMARY_PATH = PREPROCESS_DIR / "dinov2_preprocessing_summary.json"

if not PREPROCESSED_PATH.is_file():
    raise FileNotFoundError(
        f"Phase 2A NPZ not found: {PREPROCESSED_PATH}. "
        "Run notebooks/08_local_dinov2_preprocessing.ipynb first."
    )
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Phase 2A summary not found: {SUMMARY_PATH}. "
        "Re-run the save-output cell in notebook 08."
    )

with np.load(PREPROCESSED_PATH, allow_pickle=False) as data:
    images_np = data["images"].copy()
    window_valid_np = data["window_valid"].copy()
    window_positions_np = data["window_positions"].copy()
    saved_role_order = data["role_order"].astype(str).tolist()
    role_series_uids = data["role_series_uids"].astype(str).tolist()

preprocessing_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
STUDY_UID = str(preprocessing_summary["study_instance_uid"])

assert saved_role_order == list(FIVE_ROLE_ORDER), (saved_role_order, list(FIVE_ROLE_ORDER))
assert images_np.shape == (5, 10, 3, 224, 224), images_np.shape
assert images_np.dtype == np.uint8, images_np.dtype
assert window_valid_np.shape == (5, 10), window_valid_np.shape
assert window_positions_np.shape == (5, 10), window_positions_np.shape
assert np.isfinite(window_positions_np).all()

N_VALID_WINDOWS = int(window_valid_np.astype(bool).sum())
N_AVAILABLE_ROLES = int(window_valid_np.astype(bool).any(axis=1).sum())
print("Study UID:", STUDY_UID)
print("Preprocessed cache:", PREPROCESSED_PATH)
print("Images:", images_np.shape, images_np.dtype)
print("Valid windows:", N_VALID_WINDOWS, "/ 50")
print("Available roles:", N_AVAILABLE_ROLES, "/ 5")
for role, series_uid in zip(FIVE_ROLE_ORDER, role_series_uids):
    print(f"  {role}: {series_uid}")

if N_VALID_WINDOWS == 0:
    raise RuntimeError(
        "Phase 2A cache has zero valid windows. Recheck DICOM preprocessing before model inference."
    )

Study UID: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
Preprocessed cache: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_preprocessing\dinov2_preprocessed_sample.npz
Images: (5, 10, 3, 224, 224) uint8
Valid windows: 50 / 50
Available roles: 5 / 5
  SAG_FLUID: 1.2.826.0.1.3680043.8.498.37372672584387115708078590109474719429
  COR_FLUID: 1.2.826.0.1.3680043.8.498.21497837050858589852101556928971904198
  AX_FLUID: 1.2.826.0.1.3680043.8.498.67495435033389756963800730529101229430
  SAG_NONFLUID: 1.2.826.0.1.3680043.8.498.10791858932231443712736629517371931282
  COR_NONFLUID: 1.2.826.0.1.3680043.8.498.12451507871565834868863189854063726441


## 3. Resolve and validate the final five-fold DINOv2 bundle

Do not point this notebook to the older generic `dinov2-artifact.zip`. It requires the final bundle used by the competition submission.

In [3]:
# Set explicitly if automatic discovery cannot find your final bundle.
# Example: PROJECT_ROOT / "artifacts" / "dinov2" / "five_role_v2_cv5_submission_bundle"
# Or the complete final ZIP ending in five-role-v2-cv5-submission-bundle.zip.
DINO_BUNDLE_SOURCE = None

if DINO_BUNDLE_SOURCE is None:
    artifact_root = PROJECT_ROOT / "artifacts"
    if not artifact_root.exists():
        raise FileNotFoundError(
            f"Artifact folder not found: {artifact_root}. Set DINO_BUNDLE_SOURCE explicitly."
        )
    try:
        bundle_root = find_bundle_root(artifact_root)
    except Exception as directory_error:
        zip_candidates = list(artifact_root.rglob("*five*role*cv5*submission*bundle*.zip"))
        if len(zip_candidates) == 1:
            extraction_dir = artifact_root / "_extracted_final_dinov2_bundle"
            bundle_root = resolve_bundle_root(zip_candidates[0], extraction_dir)
        else:
            raise RuntimeError(
                "Could not auto-discover exactly one complete final five-fold DINOv2 bundle. "
                "Set DINO_BUNDLE_SOURCE to its extracted directory or complete ZIP."
            ) from directory_error
else:
    bundle_source = Path(DINO_BUNDLE_SOURCE).expanduser().resolve()
    extraction_dir = PROJECT_ROOT / "artifacts" / "_extracted_final_dinov2_bundle"
    bundle_root = resolve_bundle_root(
        bundle_source,
        extraction_dir if bundle_source.suffix.lower() == ".zip" else None,
    )

bundle_root, bundle_manifest = validate_bundle_root(bundle_root)
assert list(bundle_manifest["targets"]) == list(TARGETS)
assert list(bundle_manifest["role_order"]) == list(FIVE_ROLE_ORDER)

print("Resolved bundle:", bundle_root)
print("Architecture:", bundle_manifest["architecture"])
print("Backbone:", bundle_manifest["backbone_model"])
print("Target count:", len(bundle_manifest["targets"]))
print("Role order:", bundle_manifest["role_order"])
for fold in range(1, 6):
    checkpoint_path = (
        bundle_root / "five_role_2p5d_v2_cv5" / f"fold_{fold}" / "best_checkpoint.pt"
    )
    print(
        f"Fold {fold}: exists={checkpoint_path.is_file()} | "
        f"size={checkpoint_path.stat().st_size / 1024**2:.1f} MiB"
    )

Resolved bundle: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle
Architecture: DINOv2 ViT-S/14 shared 2.5D five-role; target-by-role attention; 12 independent heads
Backbone: dinov2_vits14
Target count: 12
Role order: ['SAG_FLUID', 'COR_FLUID', 'AX_FLUID', 'SAG_NONFLUID', 'COR_NONFLUID']
Fold 1: exists=True | size=93.0 MiB
Fold 2: exists=True | size=93.0 MiB
Fold 3: exists=True | size=93.0 MiB
Fold 4: exists=True | size=93.0 MiB
Fold 5: exists=True | size=93.0 MiB


## 4. Run full five-fold inference

Each fold processes every valid window in the study. The fold model is deleted before the next checkpoint is loaded. The aggregation is the mean of the **five sigmoid probability vectors**, matching Kaggle Cell 33.

In [4]:
# For a memory-constrained CPU, try 1. This affects encoder batching only.
# Do not change preprocessing, weights, target order or role order.
# ENCODER_CHUNK_SIZE = 1

inference_started = time.perf_counter()
result = run_dinov2_five_fold_inference(
    images=images_np,
    window_valid=window_valid_np,
    window_positions=window_positions_np,
    bundle_root=bundle_root,
    study_uid=STUDY_UID,
    device=DEVICE,
    encoder_chunk_size=ENCODER_CHUNK_SIZE,
    use_amp=USE_AMP,
    verbose=True,
)

assert result.status == "ok", result.status
assert result.fold_probabilities.shape == (5, len(TARGETS))
assert result.ensemble_probabilities.shape == (len(TARGETS),)
assert len(result.fold_info) == 5
assert all(item["strict_load"] is True for item in result.fold_info)
assert np.isfinite(result.fold_probabilities).all()
assert np.isfinite(result.ensemble_probabilities).all()
assert np.all((result.fold_probabilities >= 0) & (result.fold_probabilities <= 1))
assert np.all((result.ensemble_probabilities >= 0) & (result.ensemble_probabilities <= 1))

print("\nFULL FIVE-FOLD INFERENCE: PASS")
print("Status:", result.status)
print("Study:", result.study_uid)
print("Device:", result.device)
print("Valid windows encoded per fold:", result.n_valid_windows, "/ 50")
print("Roles available:", result.n_available_roles, "/ 5")
print("Total elapsed seconds:", round(result.elapsed_seconds, 2))

C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\_extracted_final_dinov2_bundle\dinov2_repo\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Fold 1/5: strict_load=True | encoded_windows=50/50 | range=[0.2325, 0.5401] | elapsed=2.6s
Fold 2/5: strict_load=True | encoded_windows=50/50 | range=[0.2160, 0.6091] | elapsed=2.6s
Fold 3/5: strict_load=True | encoded_windows=50/50 | range=[0.0803, 0.5840] | elapsed=2.6s
Fold 4/5: strict_load=True | encoded_windows=50/50 | range=[0.2497, 0.6498] | elapsed=4.3s
Fold 5/5: strict_load=True | encoded_windows=50/50 | range=[0.0377, 0.6263] | elapsed=4.3s
Five-fold inference complete | study=1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149 | valid_windows=50/50 | available_roles=5/5 | elapsed=17.0s

FULL FIVE-FOLD INFERENCE: PASS
Status: ok
Study: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
Device: cpu
Valid windows encoded per fold: 50 / 50
Roles available: 5 / 5
Total elapsed seconds: 17.02


## 5. Inspect fold outputs and five-fold mean

In [5]:
fold_table = pd.DataFrame(result.fold_probabilities, columns=TARGETS)
fold_table.insert(0, "fold", [f"fold_{i}" for i in range(1, 6)])
display(fold_table)

target_table = pd.DataFrame({
    "target": TARGETS,
    **{f"fold_{i}": result.fold_probabilities[i - 1] for i in range(1, 6)},
    "ensemble_mean": result.ensemble_probabilities,
    "fold_std": result.fold_std,
})
display(target_table)

# Explicitly verify probability-space averaging (not logit averaging).
manual_mean = result.fold_probabilities.mean(axis=0)
np.testing.assert_allclose(result.ensemble_probabilities, manual_mean, rtol=1e-6, atol=1e-7)
print("Probability-space mean verification: PASS")
print("Ensemble probability range:",
      float(result.ensemble_probabilities.min()),
      "to",
      float(result.ensemble_probabilities.max()))

,fold,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,fold_1,0.232488,0.346627,0.428930,0.408395,0.540146,0.399790,0.421094,0.328672,0.310544,0.354817,0.454500,0.348161
1,fold_2,0.301422,0.263486,0.216017,0.349939,0.385235,0.549795,0.549001,0.609147,0.416139,0.328890,0.313571,0.243023
2,fold_3,0.080290,0.225248,0.234410,0.116267,0.286969,0.307702,0.544997,0.584024,0.286599,0.241199,0.281339,0.109193
3,fold_4,0.452808,0.529137,0.288849,0.310216,0.397763,0.537637,0.530647,0.649773,0.455872,0.393651,0.499757,0.249747
4,fold_5,0.114674,0.089175,0.077358,0.133228,0.249984,0.492480,0.626333,0.465677,0.135824,0.142943,0.054462,0.037692


,target,fold_1,fold_2,fold_3,fold_4,fold_5,ensemble_mean,fold_std
0,ACL,0.232488,0.301422,0.080290,0.452808,0.114674,0.236336,0.134360
1,MCL,0.346627,0.263486,0.225248,0.529137,0.089175,0.290734,0.145339
2,Medial Meniscus,0.428930,0.216017,0.234410,0.288849,0.077358,0.249113,0.113776
3,Lateral Meniscus,0.408395,0.349939,0.116267,0.310216,0.133228,0.263609,0.117726
4,Medial OA,0.540146,0.385235,0.286969,0.397763,0.249984,0.372019,0.101223
5,Lateral OA,0.399790,0.549795,0.307702,0.537637,0.492480,0.457481,0.091579
6,PF OA,0.421094,0.549001,0.544997,0.530647,0.626333,0.534415,0.065772
7,Effusion,0.328672,0.609147,0.584024,0.649773,0.465677,0.527459,0.116762
8,Synovitis,0.310544,0.416139,0.286599,0.455872,0.135824,0.320996,0.112094
9,Baker's,0.354817,0.328890,0.241199,0.393651,0.142943,0.292300,0.089931


Probability-space mean verification: PASS
Ensemble probability range: 0.19756349921226501 to 0.5344145894050598


## 6. Save inference artifacts

In [6]:
OUTPUT_DIR = PROJECT_ROOT / "demo" / "output" / "dinov2_full_inference"
saved_paths = save_dinov2_inference_result(result, OUTPUT_DIR)

print("Saved DINOv2 full-inference outputs:")
for name, path in saved_paths.items():
    print(f"  {name}: {path}")

assert all(path.is_file() for path in saved_paths.values())
print("Output serialization: PASS")

Saved DINOv2 full-inference outputs:
  fold_probabilities_csv: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_full_inference\dinov2_fold_probabilities.csv
  ensemble_prediction_csv: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_full_inference\dinov2_ensemble_prediction.csv
  predictions_npz: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_full_inference\dinov2_five_fold_predictions.npz
  report_json: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_full_inference\dinov2_five_fold_inference_report.json
Output serialization: PASS


## 7. Phase 2C completion criteria

In [7]:
criteria = {
    "Phase 2A input contract": (
        images_np.shape == (5, 10, 3, 224, 224)
        and images_np.dtype == np.uint8
        and window_valid_np.shape == (5, 10)
        and window_positions_np.shape == (5, 10)
    ),
    "Final five-fold bundle validated": True,
    "All five folds restored with strict=True": (
        len(result.fold_info) == 5
        and all(info.get("strict_load") is True for info in result.fold_info)
    ),
    "Full-study windows processed in every fold": all(
        info.get("valid_windows_encoded") == result.n_valid_windows
        for info in result.fold_info
    ),
    "Each fold returned 12 finite probabilities": result.fold_probabilities.shape == (5, 12)
        and np.isfinite(result.fold_probabilities).all(),
    "Five-fold probability mean verified": np.allclose(
        result.ensemble_probabilities,
        result.fold_probabilities.mean(axis=0),
        rtol=1e-6,
        atol=1e-7,
    ),
    "All output files saved": all(path.is_file() for path in saved_paths.values()),
    "Inference status is successful": result.status == "ok",
}

for name, passed in criteria.items():
    print(f"{'[PASS]' if passed else '[FAIL]'} {name}")

assert all(criteria.values()), "Phase 2C did not pass every completion criterion."
print("\nPHASE 2C: ALL CRITERIA PASSED")

[PASS] Phase 2A input contract
[PASS] Final five-fold bundle validated
[PASS] All five folds restored with strict=True
[PASS] Full-study windows processed in every fold
[PASS] Each fold returned 12 finite probabilities
[PASS] Five-fold probability mean verified
[PASS] All output files saved
[PASS] Inference status is successful

PHASE 2C: ALL CRITERIA PASSED
